# Merchant ranking: risk-adjusted expected BNPL profit

Ranks the 4,026 onboarding candidates by what the BNPL firm expects to **earn** from each over the next 12 weeks:

All code is in this notebook: helper functions in section 0, analysis after.

## Helper Functions

In [22]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import pyarrow.dataset as ds
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import ndcg_score

TX_PARTS = ["part_2", "part_3", "part_4"]

In [23]:
# loading
def find_project_root(start=None) -> Path:
    cur = Path(start or Path.cwd()).resolve()
    for p in [cur, *cur.parents]:
        if (p / "data" / "tables" / "part_1").exists():
            return p
    raise FileNotFoundError(f"Could not find data/tables/part_1 in {cur} or any parent folder")


def load_transactions(tables_dir: Path) -> pd.DataFrame:
    curated = tables_dir.parent / "curated" / "curated_transactions"
    frames = []
    for part in TX_PARTS:
        df = pd.read_parquet(curated / f"{part}.parquet",
                             columns=["user_id", "merchant_abn", "dollar_value", "order_datetime"])
        df["order_datetime"] = pd.to_datetime(df["order_datetime"])
        frames.append(df)
    tx = pd.concat(frames, ignore_index=True)
    tx["user_id"] = tx["user_id"].astype("int32")
    tx["dollar_value"] = tx["dollar_value"].astype("float64")
    return tx

TEAM_SEGMENT_KEYWORDS = {
    'Fashion & Lifestyle': [
        'shoe', 'clothing', 'apparel', 'fashion', 'jewel', 'watch', 'gift', 'novelty',
        'souvenir', 'florist', 'flower', 'beauty', 'cosmetic', 'accessor', 'leather',
        'luggage', 'department store', 'sporting goods', 'toy', 'game shop',
    ],
    'Electronics & Digital': [
        'digital goods', 'computer', 'software', 'telecom', 'telephone', 'electronic',
        'cable', 'satellite', 'television', 'radio services', 'camera', 'photographic',
        'internet', 'data processing', 'programming', 'appliance',
    ],
    'Home & Living': [
        'furniture', 'home furnishing', 'household', 'hardware', 'building material',
        'garden', 'nursery stock', 'floor covering', 'carpet', 'interior', 'lighting',
        'kitchen', 'home improvement', 'paint', 'decor',
    ],
    'Automotive & Outdoor': [
        'motor vehicle', 'automotive', 'automobile', 'auto part', 'bicycle', 'tent',
        'awning', 'camping', 'marine', 'boat', 'motorcycle', 'tyre', 'tire',
        'recreational vehicle', 'outdoor',
    ],
    'Arts, Media & Services': [
        'art dealer', 'art galler', 'book', 'periodical', 'newspaper', 'music',
        'instrument', 'sheet music', 'entertainment', 'cinema', 'theatre', 'theater',
        'professional service', 'repair', 'education', 'travel', 'hotel',
        'restaurant', 'cafe', 'food', 'health', 'medical', 'optician', 'optical',
    ],
}
TEAM_SEGMENTS = list(TEAM_SEGMENT_KEYWORDS)          # the five ranked segments
ALL_SEGMENTS = TEAM_SEGMENTS + ["Other"]


def assign_team_segment(industry):
    if pd.isna(industry):
        return 'Unknown'
    for segment, keywords in TEAM_SEGMENT_KEYWORDS.items():
        if any(keyword in industry for keyword in keywords):
            return segment
    return 'Other'


SEGMENT_MAP_4 = {
    "Electronics & Digital": [
        "computers, computer peripheral equipment, and software",
        "computer programming , data processing, and integrated systems design services",
        "telecom",
        "cable, satellite, and other pay television and radio services",
        "digital goods: books, movies, music",
    ],
    "Home, Garden & Auto": [
        "furniture, home furnishings and equipment shops, and manufacturers, except appliances",
        "equipment, tool, furniture, and appliance rent al and leasing",
        "lawn and garden supply outlets, including nurseries",
        "florists supplies, nursery stock, and flowers",
        "tent and awning shops",
        "motor vehicle supplies and new parts",
    ],
    "Hobby, Books & Leisure": [
        "books, periodicals, and newspapers",
        "music shops - musical instruments, pianos, and sheet music",
        "hobby, toy and game shops",
        "artist supply and craft shops",
        "art dealers and galleries",
        "antique shops - sales, repairs, and restoration services",
        "bicycle shops - sales and service",
        "gift, card, novelty, and souvenir shops",
        "stationery, office supplies and printing and writing paper",
    ],
    "Personal & Luxury": [
        "jewelry, watch, clock, and silverware shops",
        "watch, clock, and jewelry repair shops",
        "opticians, optical goods, and eyeglasses",
        "shoe shops",
        "health and beauty spas",
    ],
}
CATEGORY_TO_SEGMENT_4 = {c: s for s, cats in SEGMENT_MAP_4.items() for c in cats}

_TAG_RE = re.compile(
    r"^[\(\[]\s*[\(\[]\s*(?P<cat>.*?)\s*[\)\]]\s*,\s*[\(\[]\s*(?P<band>[a-e])\s*[\)\]]\s*,"
    r"\s*[\(\[]\s*take rate:\s*(?P<rate>[\d.]+)\s*[\)\]]\s*[\)\]]$")


def load_merchants(tables_dir: Path) -> pd.DataFrame:
    m = pd.read_parquet(tables_dir / "part_1" / "tbl_merchants.parquet").reset_index()
    tags = m["tags"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
    parsed = tags.str.extract(_TAG_RE)
    assert parsed.notna().all().all(), "unparsed merchant tags"
    m["category"] = parsed["cat"]
    m["revenue_band"] = parsed["band"]
    m["take_rate"] = parsed["rate"].astype(float) / 100          # fraction
    m["segment"] = m["category"].map(assign_team_segment)
    m["segment_4"] = m["category"].map(CATEGORY_TO_SEGMENT_4)
    assert m["segment"].isin(ALL_SEGMENTS).all()
    return m[["merchant_abn", "name", "category", "segment", "segment_4", "revenue_band", "take_rate"]]


def load_fraud_labels(tables_dir: Path):
    cf = pd.read_csv(tables_dir / "part_1" / "consumer_fraud_probability.csv").drop_duplicates()
    mf = pd.read_csv(tables_dir / "part_1" / "merchant_fraud_probability.csv").drop_duplicates()
    for f in (cf, mf):
        f["order_datetime"] = pd.to_datetime(f["order_datetime"])
        f["fraud_probability"] = f["fraud_probability"] / 100
    assert not cf.duplicated(["user_id", "order_datetime"]).any()
    assert not mf.duplicated(["merchant_abn", "order_datetime"]).any()
    label_end = max(cf["order_datetime"].max(), mf["order_datetime"].max())
    return cf, mf, label_end


def attach_fraud(tx: pd.DataFrame, cf: pd.DataFrame, mf: pd.DataFrame) -> pd.DataFrame:
    tx = tx.merge(cf.rename(columns={"fraud_probability": "p_consumer"}),
                  on=["user_id", "order_datetime"], how="left", validate="many_to_one")
    tx = tx.merge(mf.rename(columns={"fraud_probability": "p_merchant"}),
                  on=["merchant_abn", "order_datetime"], how="left", validate="many_to_one")
    pc, pm = tx.pop("p_consumer").fillna(0.0), tx.pop("p_merchant").fillna(0.0)
    tx["p_fraud"] = 1 - (1 - pc) * (1 - pm)
    tx["fraud_dollars"] = tx["dollar_value"] * tx["p_fraud"]
    return tx

In [24]:
# Weekly Panel
# Weeks run from Monday to Sunday, where Sunday is the origin
def weekly_matrix(tx: pd.DataFrame, merchants: pd.Index, col: str = "dollar_value") -> pd.DataFrame:
    week = tx["order_datetime"].dt.to_period("W-SUN").dt.start_time
    mat = tx.groupby([tx["merchant_abn"], week])[col].sum().unstack(fill_value=0.0)
    first, last = tx["order_datetime"].min(), tx["order_datetime"].max()
    complete = [w for w in mat.columns if w >= first.normalize() and w + pd.Timedelta(days=6) <= last.normalize()]
    return mat.reindex(index=merchants, columns=complete, fill_value=0.0)


def _window(mat: pd.DataFrame, T: pd.Timestamp, weeks: int, offset: int = 0) -> pd.Series:
    end = T - pd.Timedelta(weeks=offset)
    start = end - pd.Timedelta(weeks=weeks) + pd.Timedelta(days=1)
    cols = [w for w in mat.columns if w >= start.normalize() and w + pd.Timedelta(days=6) <= end]
    return mat[cols].sum(axis=1)


def future_sum(mat: pd.DataFrame, T: pd.Timestamp, weeks: int) -> pd.Series:
    cols = [w for w in mat.columns if T < w and w + pd.Timedelta(days=6) <= T + pd.Timedelta(weeks=weeks)]
    assert len(cols) == weeks, f"future window after {T.date()} has {len(cols)} of {weeks} weeks"
    return mat[cols].sum(axis=1)

In [25]:
def fraud_share_asof(tx: pd.DataFrame, merchants: pd.DataFrame, T, label_end, prior_mult=1.0,
                     prior="segment_aov", n_aov_buckets=5):
    cut = min(pd.Timestamp(T), label_end)
    h = tx[tx["order_datetime"] <= cut]
    g = h.groupby("merchant_abn").agg(dollar_value=("dollar_value", "sum"),
                                      fraud_dollars=("fraud_dollars", "sum"),
                                      n_tx=("dollar_value", "size"))
    df = merchants[["merchant_abn", "segment"]].set_index("merchant_abn").join(g).fillna(0.0)
    seg = df.groupby("segment")[["dollar_value", "fraud_dollars"]].sum()
    seg_share = df["segment"].map(seg["fraud_dollars"] / seg["dollar_value"])
    if prior == "segment_aov":
        aov = (df["dollar_value"] / df["n_tx"]).where(df["n_tx"] > 0)
        q = pd.qcut(aov.rank(method="first"), n_aov_buckets, labels=False)
        bucket = df["segment"] + "|" + q.astype("Int64").astype(str)
        b = df.groupby(bucket)[["dollar_value", "fraud_dollars"]].sum()
        prior_share = bucket.map(b["fraud_dollars"] / b["dollar_value"]).where(aov.notna()).fillna(seg_share)
    else:
        prior_share = seg_share
    kk = df["segment"].map(df.groupby("segment")["dollar_value"].median() * prior_mult)
    raw = (df["fraud_dollars"] / df["dollar_value"]).where(df["dollar_value"] > 0)
    shrunk = (df["fraud_dollars"] + kk * prior_share) / (df["dollar_value"] + kk)
    return pd.DataFrame({"fraud_share_raw": raw, "fraud_prior": prior_share, "fraud_share": shrunk})


def features_asof(tx, gmv_mat, merchants, T, H):
    T = pd.Timestamp(T)
    idx = merchants.set_index("merchant_abn")
    f = pd.DataFrame(index=gmv_mat.index)
    for w in (4, 12, 26):
        f[f"gmv_{w}w"] = _window(gmv_mat, T, w)
    f["gmv_prev_12w"] = _window(gmv_mat, T, 12, offset=12)
    f["growth_12w"] = np.log((f["gmv_12w"] + 1) / (f["gmv_prev_12w"] + 1))
    hist_cols = [w for w in gmv_mat.columns if w + pd.Timedelta(days=6) <= T]
    hist = gmv_mat[hist_cols]
    f["gmv_hist"] = hist.sum(axis=1)
    last26 = hist.iloc[:, -26:]
    f["weekly_sd_26w"] = last26.std(axis=1)
    f["cv_26w"] = f["weekly_sd_26w"] / last26.mean(axis=1).replace(0, np.nan)
    active = hist.gt(0)
    first_active = active.idxmax(axis=1).where(active.any(axis=1))
    f["tenure_weeks"] = ((T - first_active).dt.days / 7).fillna(0)
    f["active_share_26w"] = last26.gt(0).mean(axis=1)

    w12 = tx[(tx["order_datetime"] > T - pd.Timedelta(weeks=12)) & (tx["order_datetime"] <= T)]
    g12 = w12.groupby("merchant_abn")
    f["customers_12w"] = g12["user_id"].nunique()
    f["tx_per_week_12w"] = g12.size() / 12
    w26 = tx[(tx["order_datetime"] > T - pd.Timedelta(weeks=26)) & (tx["order_datetime"] <= T)]
    per_cust = w26.groupby(["merchant_abn", "user_id"]).size()
    f["repeat_share_26w"] = per_cust.ge(2).groupby(level=0).mean()
    f["aov_12w"] = f["gmv_12w"] / g12.size()

    f["take_rate"] = idx["take_rate"]
    f["band_ord"] = idx["revenue_band"].map({"a": 4, "b": 3, "c": 2, "d": 1, "e": 0})
    for s in ALL_SEGMENTS:
        f[f"seg_{s}"] = (idx["segment"] == s).astype(float)

    # calendar of forecast model
    mid = T + pd.Timedelta(weeks=H / 2)
    f["window_sin"], f["window_cos"] = np.sin(2 * np.pi * mid.dayofyear / 365.25), np.cos(2 * np.pi * mid.dayofyear / 365.25)
    return f.fillna({"customers_12w": 0, "tx_per_week_12w": 0, "repeat_share_26w": 0, "aov_12w": 0, "cv_26w": 0})


def _model_cols(f):
    drop = {"gmv_prev_12w"}
    return [c for c in f.columns if c not in drop]

In [26]:
# forecasters - each returns expected GMV over the next H weeks, given T
def forecast_run_rate(f, H, weeks):
    return f[f"gmv_{weeks}w"] / weeks * H


def fit_gbm(tx, gmv_mat, merchants, T, H, step_weeks=2, min_history_weeks=12, seed=42, cache=None):
    T = pd.Timestamp(T)
    first_week = gmv_mat.columns.min()
    start = first_week + pd.Timedelta(weeks=min_history_weeks) - pd.Timedelta(days=1)
    origins = []
    o = T - pd.Timedelta(weeks=H)
    while o >= start:
        origins.append(o)
        o -= pd.Timedelta(weeks=step_weeks)
    X, y = [], []
    for o in sorted(origins):
        key = (o, H)
        if cache is not None and key in cache:
            fo = cache[key]
        else:
            fo = features_asof(tx, gmv_mat, merchants, o, H)
            fo["_y"] = np.log1p(future_sum(gmv_mat, o, H))
            if cache is not None:
                cache[key] = fo
        X.append(fo)
    X = pd.concat(X)
    cols = _model_cols(X.drop(columns="_y"))
    model = HistGradientBoostingRegressor(max_iter=400, learning_rate=0.05, max_leaf_nodes=31,
                                          min_samples_leaf=40, l2_regularization=1.0, random_state=seed)
    model.fit(X[cols], X["_y"])
    resid = X["_y"] - model.predict(X[cols])
    smear = float(np.mean(np.exp(resid)))          # Duan smearing: log-scale -> mean dollars
    return model, cols, smear, len(origins), resid


def forecast_gbm(model_bundle, f):
    model, cols, smear, *_ = model_bundle
    return pd.Series(np.expm1(model.predict(f[cols])) * smear, index=f.index).clip(lower=0)

In [27]:
# Scoring and Evaluation
def score(gmv_fc, take_rate, fraud_share, sigma_gmv, lgd=1.0, lam=0.0):
    revenue = gmv_fc * take_rate
    fraud_cost = gmv_fc * fraud_share * lgd
    stability_penalty = lam * sigma_gmv * take_rate
    return pd.DataFrame({"exp_revenue": revenue, "exp_fraud_cost": fraud_cost,
                         "stability_penalty": stability_penalty,
                         "score": revenue - fraud_cost - stability_penalty})


def placeholder_score(f, fraud_share):
    p = lambda s: s.rank(pct=True)
    return (0.25 * p(f["gmv_12w"]) + 0.15 * p(f["gmv_hist"]) + 0.20 * p(f["customers_12w"])
            + 0.15 * p(f["tx_per_week_12w"]) + 0.15 * p(f["repeat_share_26w"]) - 0.10 * p(fraud_share))


PLACEHOLDER_WEIGHTS = {"predicted_profit": 0.25, "historical_value": 0.15, "unique_customers": 0.20,
                       "tx_frequency": 0.15, "repeat_share": 0.15, "fraud_risk": 0.10}


def placeholder_components(f, fraud_share, gmv_fc):
    mm = lambda s: (s - s.min()) / (s.max() - s.min())
    return pd.DataFrame({
        "predicted_profit": mm(gmv_fc * (f["take_rate"] - fraud_share)),
        "historical_value": mm(f["gmv_hist"]),
        "unique_customers": mm(f["customers_12w"]),
        "tx_frequency": mm(f["tx_per_week_12w"]),
        "repeat_share": mm(f["repeat_share_26w"]),
        "fraud_risk": mm(fraud_share),
    })


def placeholder_fixed_score(f, fraud_share, gmv_fc):
    c = placeholder_components(f, fraud_share, gmv_fc)
    w = PLACEHOLDER_WEIGHTS
    return sum(w[k] * c[k] for k in w if k != "fraud_risk") - w["fraud_risk"] * c["fraud_risk"]


def selection_metrics(score_s, realised, k=100):
    pick = score_s.nlargest(k).index
    oracle = realised.nlargest(k)
    rel = realised.clip(lower=0).to_numpy()[None, :]
    return {"capture": realised.loc[pick].sum() / oracle.sum(),
            "precision@k": len(set(pick) & set(oracle.index)) / k,
            "ndcg@k": ndcg_score(rel, score_s.to_numpy()[None, :], k=k)}


def jaccard(a, b):
    a, b = set(a), set(b)
    return len(a & b) / len(a | b)

In [28]:
# Team Fraud Model
RANDOM_STATE = 42
SMOOTHING = 10.0


def load_labels_0_100(tables_dir: Path):
    cl = pd.read_csv(tables_dir / "part_1" / "consumer_fraud_probability.csv").drop_duplicates()
    ml = pd.read_csv(tables_dir / "part_1" / "merchant_fraud_probability.csv").drop_duplicates()
    for f in (cl, ml):
        f["order_datetime"] = pd.to_datetime(f["order_datetime"].astype(str))
    return cl, ml


# how often a customer's or merchant's activity falls on a flagged day
def flag_rates(tx, cl, ml, T, label_end, weight="days"):
    cut = min(pd.Timestamp(T), label_end)
    h = tx.loc[tx["order_datetime"] <= cut, ["user_id", "merchant_abn", "order_datetime", "dollar_value"]]
    out = {}
    for key, labels in (("user_id", cl), ("merchant_abn", ml)):
        days = h.groupby([key, "order_datetime"], as_index=False)["dollar_value"].sum()
        lab = labels.loc[labels["order_datetime"] <= cut, [key, "order_datetime"]].drop_duplicates()
        lab["flag"] = 1
        days = days.merge(lab, on=[key, "order_datetime"], how="left").fillna({"flag": 0})
        w = days["dollar_value"] if weight == "dollars" else pd.Series(1.0, index=days.index)
        days["num"], days["den"] = days["flag"] * w, w
        g = days.groupby(key)[["num", "den"]].sum()
        overall = g["num"].sum() / g["den"].sum()
        k = SMOOTHING * (g["den"].median() / days.groupby(key).size().median() if weight == "dollars" else 1.0)
        out[key] = ((g["num"] + k * overall) / (g["den"] + k), overall)
    return out


FP2_FEATS = ["n_txn", "total_value", "avg_value", "max_value", "n_other", "txn_per_other",
             "dow", "month", "days_since_start"]
FP2_ENT = {"consumer": ("user_id", "merchant_abn"), "merchant": ("merchant_abn", "user_id")}

def fp2_daily(tx, key, other):
    t = tx.loc[tx["dollar_value"] >= 0.01, [key, other, "order_datetime", "dollar_value"]]
    d = t.groupby([key, "order_datetime"]).agg(n_txn=("dollar_value", "size"), total_value=("dollar_value", "sum"),
                                              avg_value=("dollar_value", "mean"), max_value=("dollar_value", "max"),
                                              n_other=(other, "nunique")).reset_index()
    d["txn_per_other"] = d["n_txn"] / d["n_other"]
    d["dow"] = (d["order_datetime"].dt.dayofweek + 1) % 7 + 1          # Spark dayofweek: Sun=1 .. Sat=7
    d["month"] = d["order_datetime"].dt.month
    d["days_since_start"] = (d["order_datetime"] - pd.Timestamp("2021-02-28")).dt.days
    return d

def fp2_model(n_rows, seed=42):
    return HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, max_leaf_nodes=31,
                                         min_samples_leaf=min(50, max(5, n_rows // 20)), l2_regularization=1.0,
                                         early_stopping=(n_rows >= 1000), random_state=seed)

def fp2_labels(labels, key, cut):
    l = labels[labels["order_datetime"] <= cut]
    return l.groupby([key, "order_datetime"], as_index=False)["fraud_probability"].mean()

def fp2_daily_all(tx):
    return {name: fp2_daily(tx, key, other) for name, (key, other) in FP2_ENT.items()}

def _merchant_share(recent_tx, p, merchants):
    r = recent_tx.assign(fd=recent_tx["dollar_value"].to_numpy() * p)
    g = r.groupby("merchant_abn")[["dollar_value", "fd"]].sum()
    idx = merchants.set_index("merchant_abn")
    s = (g["fd"] / g["dollar_value"]).reindex(idx.index)
    return s.fillna(s.groupby(idx["segment"]).transform("median"))

def fraud_share_fp2(tx, merchants, daily, cl, ml, T, label_end, lookback_weeks=26, train_weeks=12, seed=42):
    T = pd.Timestamp(T); cut = min(T, label_end)
    lo = T - pd.Timedelta(weeks=lookback_weeks)
    recent = tx.loc[(tx["order_datetime"] > lo) & (tx["order_datetime"] <= T) & (tx["dollar_value"] >= 0.01),
                    ["user_id", "merchant_abn", "order_datetime", "dollar_value"]].reset_index(drop=True)
    out = {}
    # as-is, per the model prior
    cd = daily["consumer"]
    lab = fp2_labels(cl, "user_id", cut)
    tr = cd[cd["order_datetime"] <= cut].merge(lab, on=["user_id", "order_datetime"], how="inner")
    mdl = fp2_model(len(tr), seed).fit(tr[FP2_FEATS], tr["fraud_probability"])
    sc = cd[(cd["order_datetime"] > lo) & (cd["order_datetime"] <= T)].copy()
    sc["pred"] = np.clip(mdl.predict(sc[FP2_FEATS]), 0, 100)
    sc = sc.merge(lab.rename(columns={"fraud_probability": "known"}), on=["user_id", "order_datetime"], how="left")
    sc["final"] = sc["known"].fillna(sc["pred"])
    r = recent.merge(sc[["user_id", "order_datetime", "pred", "final"]], on=["user_id", "order_datetime"], how="left")
    out["asis"] = _merchant_share(r, r["final"].fillna(r["pred"].median()).to_numpy() / 100, merchants)

    # p(flagged) x severity
    rates = flag_rates(tx, cl, ml, cut, label_end, weight="dollars")
    rc, oc = rates["user_id"]
    out["corrected"] = _merchant_share(r, r["user_id"].map(rc).fillna(oc).to_numpy() * r["pred"].fillna(r["pred"].median()).to_numpy() / 100, merchants)

    # merchant risk score
    q99 = sc["pred"].quantile(0.99)
    rr = r.assign(hr=(r["pred"] >= q99).astype(float))
    mr = rr.groupby("merchant_abn").agg(n=("final", "size"), avg=("final", "mean"), hrs=("hr", "mean"))
    mr = mr[mr["n"] >= 30]
    out["risk_score"] = ((mr["avg"].rank(pct=True) + mr["hrs"].rank(pct=True)) * 50).reindex(merchants["merchant_abn"]).values

    # adjusted 
    enc_cut = cut - pd.Timedelta(weeks=train_weeks)
    rates_tr = flag_rates(tx, cl, ml, enc_cut, label_end, weight="dollars")
    p = {}
    for name, (key, other) in FP2_ENT.items():
        d = daily[name]; labels = cl if name == "consumer" else ml
        win = d[(d["order_datetime"] > enc_cut) & (d["order_datetime"] <= cut)]
        win = win.merge(fp2_labels(labels, key, cut), on=[key, "order_datetime"], how="left").fillna({"fraud_probability": 0.0})
        r0, o0 = rates_tr[key]; win = win.assign(flag_rate=win[key].map(r0).fillna(o0))
        feats = FP2_FEATS + ["flag_rate"]
        mdl = fp2_model(len(win), seed).fit(win[feats], win["fraud_probability"])
        s = d[(d["order_datetime"] > lo) & (d["order_datetime"] <= T)]
        r1, o1 = rates[key]; s = s.assign(flag_rate=s[key].map(r1).fillna(o1))
        s = s.assign(padj=np.clip(mdl.predict(s[feats]), 0, 100) / 100)
        p[name] = recent.merge(s[[key, "order_datetime", "padj"]], on=[key, "order_datetime"], how="left")["padj"].fillna(0).to_numpy()
    out["adjusted"] = _merchant_share(recent, 1 - (1 - p["consumer"]) * (1 - p["merchant"]), merchants)
    res = pd.DataFrame({k: (v.values if hasattr(v, "values") else v) for k, v in out.items()}, index=merchants["merchant_abn"])
    return res.rename(columns={"asis": "fraud_share_asis", "corrected": "fraud_share_corrected",
                               "risk_score": "merchant_risk_score", "adjusted": "fraud_share_adjusted"})

## Setup and configuration

In [ ]:
import sys, time
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, kendalltau

NB_DIR = Path(globals().get("__vsc_ipynb_file__", Path.cwd() / "_")).resolve().parent

pd.set_option("display.width", 200, "display.max_columns", 30, "display.float_format", "{:,.4f}".format)

H = 12                      # forecast horizon in weeks 
LGD = 1.0                   # share of a fraudulent purchase the firm loses
TOP_K = 100
LAMBDAS = [0.0, 0.25, 0.5]  # stability penalty candidates, chosen by backtest
BACKTEST_ORIGINS = ["2021-10-31", "2022-01-30", "2022-05-01", "2022-07-24"]  # four non overlapping 12week windows
FRAUD_VALIDATION_ORIGINS = ["2021-08-29", "2021-10-31", "2021-11-28"]       # windows fully inside the label period
SEED = 42
N_BOOT = 300

ROOT = find_project_root(NB_DIR)
TABLES = ROOT / "data" / "tables"
ABS = ROOT / "data" / "raw_abs"
OUT = ROOT / "outputs" / "ranking"
OUT.mkdir(parents=True, exist_ok=True)
print("project root:", ROOT)

project root: /Users/leopham/Documents/dataset1_preprocessing/MAST30034_Project_2


## Data

In [30]:
merchants = load_merchants(TABLES)
tx_all = load_transactions(TABLES)
cf, mf, LABEL_END = load_fraud_labels(TABLES)
known = tx_all["merchant_abn"].isin(merchants["merchant_abn"])
print(f"transactions: {len(tx_all):,}; in merchant universe: {known.sum():,}; unknown-ABN rows dropped: {(~known).sum():,}")
tx = attach_fraud(tx_all[known].copy(), cf, mf)
del tx_all

IDX = pd.Index(merchants["merchant_abn"])
GMV = weekly_matrix(tx, IDX)
FRAUD = weekly_matrix(tx, IDX, "fraud_dollars")
FINAL_T = GMV.columns.max() + pd.Timedelta(days=6)        # last complete week from Sunday
print(f"complete weeks: {GMV.shape[1]} ({GMV.columns.min().date()} - {FINAL_T.date()}); labels end {LABEL_END.date()}")
lab = tx[tx["order_datetime"] <= LABEL_END]
print(f"expected fraud share of GMV in label period: {lab.fraud_dollars.sum() / lab.dollar_value.sum():.2%} "
      f"(vs mean take rate {merchants.take_rate.mean():.2%})")
print(f"mean order value: labelled-suspicious tx ${lab.loc[lab.p_fraud > 0, 'dollar_value'].mean():,.0f} vs all ${lab.dollar_value.mean():,.0f}")

# team fraud 
CL, ML = load_labels_0_100(TABLES)
DAILY = fp2_daily_all(tx)
print(f"consumer-days: {len(DAILY['consumer']):,}; merchant-days: {len(DAILY['merchant']):,}")

transactions: 14,195,505; in merchant universe: 13,614,675; unknown-ABN rows dropped: 580,830
complete weeks: 86 (2021-03-01 - 2022-10-23); labels end 2022-02-27
expected fraud share of GMV in label period: 2.34% (vs mean take rate 4.40%)
mean order value: labelled-suspicious tx $1,749 vs all $159
consumer-days: 8,756,431; merchant-days: 1,271,877


### Segments

In [31]:
gm = tx.groupby("merchant_abn")["dollar_value"].agg(["sum", "size"])
mj = merchants.set_index("merchant_abn").join(gm)
seg_tbl = mj.groupby("segment").agg(merchants=("name", "size"), industries=("category", "nunique"), median_take_rate=("take_rate", "median"), gmv=("sum", "sum"), orders=("size", "sum"))
seg_tbl["gmv_weighted_aov"] = seg_tbl["gmv"] / seg_tbl["orders"]
seg_tbl = seg_tbl.drop(columns="orders").reindex(ALL_SEGMENTS)
seg_tbl.to_csv(OUT / "segment_summary.csv")
seg_tbl

,merchants,industries,median_take_rate,gmv,gmv_weighted_aov
segment,,,,,
Fashion & Lifestyle,1114,7,0.0453,"555,407,320.4380",124.1980
Electronics & Digital,1183,7,0.0464,"596,424,683.0274",147.3790
Home & Living,153,1,0.0492,"108,949,190.9237",313.6799
Automotive & Outdoor,499,3,0.0452,"329,639,065.3374",140.3526
"Arts, Media & Services",723,5,0.0433,"366,971,842.3773",208.9511
Other,354,2,0.0404,"199,663,690.4411",310.2227


## Fraud Model Analysis

In [32]:
rows, t0 = [], time.time()
for o in FRAUD_VALIDATION_ORIGINS:
    T = pd.Timestamp(o)
    fut_g, fut_f = future_sum(GMV, T, H), future_sum(FRAUD, T, H)
    realised = (fut_f / fut_g).fillna(0)
    team = fraud_share_fp2(tx, merchants, DAILY, CL, ML, T, LABEL_END)
    cands = {
        "team model as-is": team["fraud_share_asis"],
        "team model ADJUSTED (USED)": team["fraud_share_adjusted"],
        "merchant history, shrunk (no model)": fraud_share_asof(tx, merchants, T, LABEL_END)["fraud_share"],
    }
    for name, est in cands.items():
        ok = est.notna()
        dollars = "risk score" not in name
        rows.append({"origin": o, "estimate": name,
                     "fraud $ abs error / realised": np.abs(fut_g * est - fut_f)[ok].sum() / fut_f.sum() if dollars else np.nan,
                     "predicted total / realised": (fut_g * est)[ok].sum() / fut_f.sum() if dollars else np.nan,
                     "spearman vs realised share": spearmanr(est[ok], realised[ok]).correlation,
                     "top-decile lift": realised[ok][est[ok] >= est[ok].quantile(0.9)].mean() / realised[ok].mean()})
    print(f"origin {o} done ({time.time() - t0:.0f}s)")
fraud_val = pd.DataFrame(rows)
fraud_val.to_csv(OUT / "fraud_estimate_validation.csv", index=False)
fraud_val.groupby("estimate", sort=False)[["fraud $ abs error / realised", "predicted total / realised",
                                           "spearman vs realised share", "top-decile lift"]].mean()

origin 2021-08-29 done (20s)
origin 2021-10-31 done (37s)
origin 2021-11-28 done (54s)


,fraud $ abs error / realised,predicted total / realised,spearman vs realised share,top-decile lift
estimate,,,,
team model as-is,3.7872,4.4160,-0.2614,3.7843
team model ADJUSTED (USED),0.6035,0.4543,0.4664,6.1558
"merchant history, shrunk (no model)",0.5909,0.4216,0.3598,5.9586


## Backtest

In [33]:
feat_cache, bt_rows, resid_pool, fraud_cache = {}, [], [], {}
rng = np.random.default_rng(SEED)
t0 = time.time()
for o in BACKTEST_ORIGINS:
    T = pd.Timestamp(o)
    f = features_asof(tx, GMV, merchants, T, H)
    key = T                                     # fraud model fitted on labels <
    if key not in fraud_cache:
        fraud_cache[key] = fraud_share_fp2(tx, merchants, DAILY, CL, ML, T, LABEL_END)
    fr = fraud_cache[key]["fraud_share_adjusted"]
    fr_hist = fraud_share_asof(tx, merchants, T, LABEL_END)["fraud_share"]
    sigma = f["weekly_sd_26w"] * np.sqrt(H)
    fut_g = future_sum(GMV, T, H)
    real_rev = fut_g * f["take_rate"]
    has_labels = T + pd.Timedelta(weeks=H) <= LABEL_END
    real_net = real_rev - future_sum(FRAUD, T, H) * LGD if has_labels else None

    gbm = fit_gbm(tx, GMV, merchants, T, H, cache=feat_cache, seed=SEED)
    forecasts = {"run_rate_12w": forecast_run_rate(f, H, 12), "gbm": forecast_gbm(gbm, f)}
    for name, fc in forecasts.items():
        resid_pool.append(pd.DataFrame({"method": name, "origin": o, "fc": fc,"log_resid": np.log1p(fut_g) - np.log1p(fc)}))
        for lam in LAMBDAS:
            s_rev = score(fc, f["take_rate"], 0.0, sigma, lgd=0.0, lam=lam)["score"]
            bt_rows.append({"origin": o, "model": name, "lambda": lam, "fraud_term": "none", **selection_metrics(s_rev, real_rev, TOP_K)})
            s_full = score(fc, f["take_rate"], fr, sigma, lgd=LGD, lam=lam)["score"]
            bt_rows.append({"origin": o, "model": name, "lambda": lam, "fraud_term": "team", **selection_metrics(s_full, real_rev, TOP_K)})
            if has_labels:
                s_hist = score(fc, f["take_rate"], fr_hist, sigma, lgd=LGD, lam=lam)["score"]
                pairs = [("team", s_full), ("history", s_hist), ("none", s_rev)]
                if (name, lam) == ("gbm", 0.0):
                    for tag, col in (("team_corrected", "fraud_share_corrected"), ("team_asis", "fraud_share_asis")):
                        pairs.append((tag, score(fc, f["take_rate"], fraud_cache[key][col], sigma, lgd=LGD, lam=lam)["score"]))
                for ft, s in pairs:
                    bt_rows.append({"origin": o, "model": name, "lambda": lam, "fraud_term": ft, "judged_on": "net profit",
                                    **selection_metrics(s, real_net, TOP_K)})
    base = {"placeholder as on slide": placeholder_score(f, fr),
            "placeholder, inputs fixed": placeholder_fixed_score(f, fr, forecasts["gbm"]),
            "trailing 12w GMV (no take rate)": f["gmv_12w"]}
    for name, s in base.items():
        bt_rows.append({"origin": o, "model": name, **selection_metrics(s, real_rev, TOP_K)})
        if has_labels:
            bt_rows.append({"origin": o, "model": name, "judged_on": "net profit", **selection_metrics(s, real_net, TOP_K)})
    rand = [selection_metrics(pd.Series(rng.random(len(f)), index=f.index), real_rev, TOP_K) for _ in range(30)]
    bt_rows.append({"origin": o, "model": "random", **pd.DataFrame(rand).mean().to_dict()})
    print(f"origin {o} done ({time.time() - t0:.0f}s)")

bt = pd.DataFrame(bt_rows)
bt["judged_on"] = bt["judged_on"].fillna("revenue")
bt.to_csv(OUT / "backtest_results.csv", index=False)
resid = pd.concat(resid_pool)

origin 2021-10-31 done (33s)
origin 2022-01-30 done (78s)
origin 2022-05-01 done (131s)
origin 2022-07-24 done (172s)


In [34]:
rev = bt[(bt.judged_on == "revenue") & (bt.fraud_term.isin(["none"]) | bt["lambda"].isna())]
summary = (rev.groupby(["model", "lambda"], dropna=False)[["capture", "precision@k", "ndcg@k"]]
              .agg(["mean", "min"]).sort_values(("capture", "mean"), ascending=False))
summary

capture        precision@k        ndcg@k       
                                          mean    min        mean    min   mean    min
model                           lambda                                                
run_rate_12w                    0.0000  0.9977 0.9958      0.9800 0.9800 0.9978 0.9967
                                0.2500  0.9976 0.9949      0.9800 0.9700 0.9977 0.9961
                                0.5000  0.9976 0.9949      0.9800 0.9700 0.9978 0.9961
gbm                             0.5000  0.9952 0.9939      0.9575 0.9500 0.9929 0.9913
                                0.2500  0.9950 0.9939      0.9575 0.9500 0.9936 0.9926
                                0.0000  0.9945 0.9936      0.9575 0.9500 0.9933 0.9924
trailing 12w GMV (no take rate) NaN     0.9639 0.9628      0.8600 0.8500 0.8957 0.8811
placeholder, inputs fixed       NaN     0.8441 0.8300      0.7225 0.7100 0.8178 0.8047
placeholder as on slide         NaN     0.6452 0.5590      0.5100 0.4400 0.6618 0.5833
random                          NaN     0.0711 0.0688      0.0248 0.0203 0.0620 0.0595

In [35]:
# forecast is the gbm; lambda by mean revenue capture across origins
cands = rev.dropna(subset=["lambda"]).groupby(["model", "lambda"])["capture"].mean().sort_values(ascending=False)
BEST_MODEL = "gbm"
BEST_LAMBDA = cands.loc[BEST_MODEL].idxmax()
RUNNER_UP_MODEL = next(m for m, _ in cands.index if m != BEST_MODEL)
print(f"chosen: forecast={BEST_MODEL}, lambda={BEST_LAMBDA}  (runner-up method: {RUNNER_UP_MODEL})")
print(cands.round(4).head(6))

chosen: forecast=gbm, lambda=0.5  (runner-up method: run_rate_12w)
model         lambda
run_rate_12w  0.0000   0.9977
              0.2500   0.9976
              0.5000   0.9976
gbm           0.5000   0.9952
              0.2500   0.9950
              0.0000   0.9945
Name: capture, dtype: float64


### Placeholder Weights v Expected Profit Model

In [ ]:
# h2h = bt[((bt.model == BEST_MODEL) & (bt["lambda"] == BEST_LAMBDA) & (bt.fraud_term == "team"))
#          | bt.model.isin(["placeholder as on slide", "placeholder, inputs fixed"])].copy()
# h2h["model"] = h2h["model"].replace({BEST_MODEL: "expected-profit model"})
# h2h_tbl = (h2h.pivot_table(index="model", columns="judged_on", values=["capture", "precision@k"], aggfunc="mean")
#               .reindex(["placeholder as on slide", "placeholder, inputs fixed", "expected-profit model"]))
# h2h_tbl.to_csv(OUT / "head_to_head_backtest.csv")
# h2h_tbl

capture         precision@k        
judged_on                 net profit revenue  net profit revenue
model                                                           
placeholder as on slide       0.8050  0.6452      0.6000  0.5100
placeholder, inputs fixed     0.8711  0.8441      0.7400  0.7225
expected-profit model         0.9492  0.8951      0.9000  0.8075

### Repeat Customer Check

In [37]:
loy = []
for o in BACKTEST_ORIGINS:
    T = pd.Timestamp(o)
    f = features_asof(tx, GMV, merchants, T, H)
    fc = resid[(resid.method == BEST_MODEL) & (resid.origin == o)]["fc"]   # gbm forecast from the backtest
    surprise = np.log1p(future_sum(GMV, T, H)) - np.log1p(fc)
    active = f["gmv_26w"] > 0
    loy.append({"origin": o,
                "spearman(repeat share, surprise)": spearmanr(f.loc[active, "repeat_share_26w"], surprise[active]).correlation,
                "spearman(customers, surprise)": spearmanr(f.loc[active, "customers_12w"], surprise[active]).correlation,
                "spearman(repeat share, 26w sales)": spearmanr(f.loc[active, "repeat_share_26w"], f.loc[active, "gmv_26w"]).correlation})
loyalty = pd.DataFrame(loy).set_index("origin")
loyalty.to_csv(OUT / "loyalty_check.csv")
loyalty

,"spearman(repeat share, surprise)","spearman(customers, surprise)","spearman(repeat share, 26w sales)"
origin,,,
2021-10-31,0.0385,-0.0489,0.6749
2022-01-30,0.0954,0.0795,0.6837
2022-05-01,0.0838,0.0206,0.6822
2022-07-24,0.1483,0.0983,0.6774


## Final Rankings

In [ ]:
T = FINAL_T
f = features_asof(tx, GMV, merchants, T, H)
fr_hist = fraud_share_asof(tx, merchants, T, LABEL_END)
fr_team = fraud_share_fp2(tx, merchants, DAILY, CL, ML, T, LABEL_END)   # scored on the final 26 weeks
FRAUD_SHARE = fr_team["fraud_share_adjusted"]
sigma = f["weekly_sd_26w"] * np.sqrt(H)
gbm_final = fit_gbm(tx, GMV, merchants, T, H, cache=feat_cache, seed=SEED)
FORECASTS = {"run_rate_12w": forecast_run_rate(f, H, 12), "gbm": forecast_gbm(gbm_final, f)}
print(f"cutoff {T.date()}: forecasting {(T + pd.Timedelta(days=1)).date()} .. {(T + pd.Timedelta(weeks=H)).date()} "
      f"with {BEST_MODEL}; GBM trained on {gbm_final[3]} origins")

def build_scores(model=BEST_MODEL, lgd=LGD, lam=BEST_LAMBDA, fraud=None, gmv_fc=None):
    fc = FORECASTS[model] if gmv_fc is None else gmv_fc
    return score(fc, f["take_rate"], FRAUD_SHARE if fraud is None else fraud, sigma, lgd=lgd, lam=lam)

S = build_scores()
ranked = (merchants.set_index("merchant_abn")[["name", "segment", "category", "revenue_band", "take_rate"]]
          .join(pd.DataFrame({"forecast_gmv_12w": FORECASTS[BEST_MODEL]}))
          .join(S)
          .join(pd.DataFrame({"fraud_share": FRAUD_SHARE, "fraud_share_team_corrected": fr_team["fraud_share_corrected"],
                              "fraud_share_team_asis": fr_team["fraud_share_asis"],
                              "team_merchant_risk_score": fr_team["merchant_risk_score"],
                              "fraud_share_history": fr_hist["fraud_share"]}))
          .join(f[["gmv_12w", "gmv_prev_12w", "growth_12w", "customers_12w", "tx_per_week_12w", "repeat_share_26w", "aov_12w", "cv_26w", "tenure_weeks", "gmv_hist"]]))
ranked["rank"] = ranked["score"].rank(ascending=False, method="first").astype(int)
in_seg = ranked["segment"].isin(TEAM_SEGMENTS)
ranked["segment_rank"] = ranked["score"].where(in_seg).groupby(ranked["segment"]).rank(ascending=False, method="first").astype("Int64")
ranked = ranked.sort_values("rank")
print(f"merchants with negative expected profit: {(ranked.score < 0).sum():,}")
hist_score = build_scores(fraud=fr_hist["fraud_share"])["score"]
print(f"top-100 overlap with the history-based fraud estimate: {len(set(ranked.index[:TOP_K]) & set(hist_score.nlargest(TOP_K).index))}")

cutoff 2022-10-23: forecasting 2022-10-24 .. 2023-01-15 with gbm; GBM trained on 32 origins
merchants with negative expected profit: 747
top-100 overlap with the history-based fraud estimate: 95


## Contrasting at the Final Cutoff With Placeholders

In [39]:
# comp = placeholder_components(f, FRAUD_SHARE, FORECASTS[BEST_MODEL]).add_prefix("ph_")
# ranked = ranked.join(comp)
# ranked["placeholder_score"] = placeholder_fixed_score(f, FRAUD_SHARE, FORECASTS[BEST_MODEL])
# ranked["placeholder_rank"] = ranked["placeholder_score"].rank(ascending=False, method="first").astype(int)

# top_p, top_ph = ranked[ranked["rank"] <= TOP_K], ranked[ranked["placeholder_rank"] <= TOP_K]
# def summarise(t):
#     return pd.Series({"expected profit ($, 12w)": t["score"].sum(), "expected revenue ($)": t["exp_revenue"].sum(),
#                       "expected fraud cost ($)": t["exp_fraud_cost"].sum(), "forecast GMV ($)": t["forecast_gmv_12w"].sum(),
#                       "mean take rate": t["take_rate"].mean(), "mean fraud share": t["fraud_share"].mean(),
#                       "merchants with negative expected profit": (t["score"] < 0).sum(),
#                       "median customers (12w)": t["customers_12w"].median(), "median repeat share": t["repeat_share_26w"].median()})
# side_summary = pd.DataFrame({"team placeholder (inputs fixed)": summarise(top_ph), "expected-profit model": summarise(top_p)})
# overlap = len(set(top_p.index) & set(top_ph.index))
# side_summary.loc["overlap with the other top 100"] = overlap
# side_summary.to_csv(OUT / "side_by_side_summary.csv")
# print(f"{overlap} merchants are in both top 100s")
# side_summary

## Identifying Robustness

In [ ]:
base_top = set(ranked.index[:TOP_K])
FRAUD_ALT = {"team model (adjusted)": FRAUD_SHARE, "merchant history": fr_hist["fraud_share"]}
scen_rows = []
for lgd in (0.0, 0.5, 1.0):
    for fname, fshare in FRAUD_ALT.items():
        for model in (BEST_MODEL, RUNNER_UP_MODEL):
            for lam in (0.0, 0.25):
                s = build_scores(model=model, lgd=lgd, lam=lam, fraud=fshare)["score"]
                top = set(s.nlargest(TOP_K).index)
                u = list(base_top | top)
                scen_rows.append({"lgd": lgd, "fraud": fname, "model": model, "lambda": lam, "jaccard_vs_base": jaccard(base_top, top), "overlap": len(base_top & top),
                                  "kendall_tau_union": kendalltau(ranked.loc[u, "score"], s.loc[u]).correlation, "_top": top})
scen = pd.DataFrame(scen_rows)
in_scen = pd.Series({abn: np.mean([abn in t for t in scen["_top"]]) for abn in ranked.index})
scen.drop(columns="_top").to_csv(OUT / "robustness_scenarios.csv", index=False)
print(f"{len(scen)} scenarios")
display(scen.drop(columns="_top").describe().loc[["mean", "min", "max"], ["overlap", "jaccard_vs_base", "kendall_tau_union"]])

pd.concat({k: scen.groupby(k)["overlap"].mean() for k in ["lgd", "fraud", "model", "lambda"]}).rename("mean overlap").to_frame()

24 scenarios


,overlap,jaccard_vs_base,kendall_tau_union
mean,90.1667,0.8273,0.7523
min,81.0000,0.6807,0.5214
max,100.0000,1.0000,0.9984


mean overlap
lgd    0.0000                       81.5000
       0.5000                       92.5000
       1.0000                       96.5000
fraud   merchant history            89.1667
        team model (adjusted)       91.1667
model   gbm                         90.1667
        run_rate_12w                90.1667
lambda 0.0000                       90.1667
       0.2500                       90.1667

In [ ]:
r = resid[resid.method == BEST_MODEL].copy()
edges = np.unique(np.quantile(np.log1p(r["fc"]), np.linspace(0, 1, 11)))
r["bin"] = np.clip(np.searchsorted(edges, np.log1p(r["fc"]), side="right") - 1, 0, len(edges) - 2)
pools = {b: g["log_resid"].to_numpy() for b, g in r.groupby("bin")}
fc0 = FORECASTS[BEST_MODEL]
bins = np.clip(np.searchsorted(edges, np.log1p(fc0), side="right") - 1, 0, len(edges) - 2)
hits = pd.Series(0, index=fc0.index, dtype=float)
boot_jacc = []
for _ in range(N_BOOT):
    e = np.empty(len(bins))
    for b, pool in pools.items():
        m = bins == b
        e[m] = rng.choice(pool, size=m.sum())
    fc_b = np.expm1(np.log1p(fc0) + e).clip(lower=0)
    top = build_scores(gmv_fc=pd.Series(fc_b, index=fc0.index))["score"].nlargest(TOP_K).index
    hits[top] += 1
    boot_jacc.append(jaccard(base_top, top))


    
ranked["p_top100_bootstrap"] = (hits / N_BOOT).reindex(ranked.index)
ranked["share_of_scenarios_top100"] = in_scen.reindex(ranked.index)
ranked["robustness"] = np.select(
    [(ranked.p_top100_bootstrap >= 0.9) & (ranked.share_of_scenarios_top100 >= 0.9),
     (ranked.p_top100_bootstrap >= 0.5)], ["core", "likely"], "edge")
print(f"bootstrap Jaccard vs base top 100: mean {np.mean(boot_jacc):.2f}, 5th pct {np.quantile(boot_jacc, 0.05):.2f}")
print(ranked.head(TOP_K)["robustness"].value_counts().reindex(["core", "likely", "edge"]).to_string())

bootstrap Jaccard vs base top 100: mean 0.79, 5th pct 0.74
robustness
core      73
likely    20
edge       7


## Outputs

In [ ]:
COLS = ["rank", "segment_rank", "name", "segment", "category", "score", "exp_revenue", "exp_fraud_cost", "stability_penalty",
        "forecast_gmv_12w", "take_rate", "fraud_share", "fraud_share_team_corrected", "fraud_share_team_asis", "team_merchant_risk_score", "fraud_share_history",
        "p_top100_bootstrap", "share_of_scenarios_top100", "robustness",
        "gmv_12w", "growth_12w", "customers_12w", "tx_per_week_12w", "repeat_share_26w", "aov_12w", "cv_26w", "tenure_weeks"]
      #   "placeholder_rank", "placeholder_score"] + [c for c in ranked.columns if c.startswith("ph_")]
out = ranked[COLS].rename_axis("merchant_abn").reset_index()
# out.head(TOP_K).to_csv(OUT / "top100_detailed.csv", index=False)
out.to_csv(OUT / "all_merchants_scored.csv", index=False)
top10_seg = out[out.segment_rank.le(10).fillna(False)].sort_values(["segment", "segment_rank"])
# top10_seg.to_csv(OUT / "top10_per_segment_detailed.csv", index=False)

# the two deliverables
CLEAN = {"rank": "rank", "segment_rank": "segment_rank", "merchant_abn": "merchant_abn", "name": "name", "segment": "segment",
         "category": "industry", "score": "expected_profit_12w", "exp_revenue": "expected_revenue_12w",
         "exp_fraud_cost": "expected_fraud_cost_12w", "forecast_gmv_12w": "forecast_sales_12w", "take_rate": "take_rate",
         "fraud_share": "fraud_share_adjusted_model", "robustness": "robustness", "customers_12w": "customers_last_12w",
         "repeat_share_26w": "repeat_customer_share_26w"}
def clean(d):
    d = d[list(CLEAN)].rename(columns=CLEAN)
    money = [c for c in d if c.startswith(("expected_", "forecast_"))]
    d[money] = d[money].round(2)
    d[["take_rate", "fraud_share_adjusted_model", "repeat_customer_share_26w"]] = d[["take_rate", "fraud_share_adjusted_model", "repeat_customer_share_26w"]].round(5)
    return d
clean(out.head(TOP_K)).to_csv(OUT / "top100_merchants.csv", index=False)
clean(top10_seg).to_csv(OUT / "top10_per_segment.csv", index=False)

top = out.head(TOP_K)
print(f"top 100: expected profit ${top.score.sum():,.0f} = revenue ${top.exp_revenue.sum():,.0f} - fraud ${top.exp_fraud_cost.sum():,.0f}"
      f" on forecast GMV ${top.forecast_gmv_12w.sum():,.0f}")
print(f"mean take rate top 100 {top.take_rate.mean():.2%} vs all {out.take_rate.mean():.2%}; "
      f"take rates {top.take_rate.min():.1%}-{top.take_rate.max():.1%}; max fraud share {top.fraud_share.max():.2%}")
print("\nTop 100 by segment:")
print(top.segment.value_counts().to_string())
show = ["rank", "name", "segment", "score", "exp_revenue", "exp_fraud_cost", "forecast_gmv_12w", "take_rate", "fraud_share", "robustness"]
out.head(20)[show]

top 100: expected profit $7,687,926 = revenue $8,468,935 - fraud $718,516 on forecast GMV $175,643,193
mean take rate top 100 5.00% vs all 4.40%; take rates 1.5%-7.0%; max fraud share 3.76%

Top 100 by segment:
segment
Fashion & Lifestyle       29
Electronics & Digital     23
Arts, Media & Services    16
Automotive & Outdoor      16
Other                     11
Home & Living              5


,rank,name,segment,score,exp_revenue,exp_fraud_cost,forecast_gmv_12w,take_rate,fraud_share,robustness
0,1,Lacus Consulting,Fashion & Lifestyle,"155,976.0786","159,102.5332","2,437.2725","2,279,405.9198",0.0698,0.0011,core
1,2,Dignissim Maecenas Foundation,"Arts, Media & Services","147,913.1844","151,775.7126","2,913.1362","2,285,778.8041",0.0664,0.0013,core
2,3,Orci In Consequat Corporation,Fashion & Lifestyle,"147,106.4224","151,089.9790","3,023.0290","2,285,778.8041",0.0661,0.0013,core
3,4,Leo In Consulting,Fashion & Lifestyle,"143,900.8935","146,975.5771","2,252.8589","2,285,778.8041",0.0643,0.0010,core
4,5,Gravida Mauris Incorporated,Fashion & Lifestyle,"141,277.4238","145,102.4900","2,938.2286","2,285,078.5822",0.0635,0.0013,core
5,6,Lobortis Ultrices Company,"Arts, Media & Services","140,646.4105","144,188.4585","2,555.6951","2,285,078.5822",0.0631,0.0011,core
6,7,Mauris Non Institute,Electronics & Digital,"134,794.4839","139,389.7935","3,714.1896","2,285,078.5822",0.0610,0.0016,core
7,8,Est Nunc Consulting,Automotive & Outdoor,"133,788.1903","137,333.2228","2,737.0653","2,285,078.5822",0.0601,0.0012,core
8,9,Odio Phasellus Institute,Other,"130,671.7800","149,536.7284","17,327.4823","2,269,146.1059",0.0659,0.0076,core
9,10,Non Vestibulum Industries,Automotive & Outdoor,"129,124.6851","132,165.0437","2,491.3146","2,278,707.6501",0.0580,0.0011,core


In [43]:
top10_seg[["segment", "segment_rank", "rank", "name", "score", "exp_revenue", "exp_fraud_cost", "take_rate", "fraud_share", "robustness"]]

,segment,segment_rank,rank,name,score,exp_revenue,exp_fraud_cost,take_rate,fraud_share,robustness
1,"Arts, Media & Services",1,2,Dignissim Maecenas Foundation,"147,913.1844","151,775.7126","2,913.1362",0.0664,0.0013,core
5,"Arts, Media & Services",2,6,Lobortis Ultrices Company,"140,646.4105","144,188.4585","2,555.6951",0.0631,0.0011,core
25,"Arts, Media & Services",3,26,Sodales At LLC,"99,634.7301","107,855.7091","7,548.1201",0.0472,0.0033,core
32,"Arts, Media & Services",4,33,Tellus Aenean Corporation,"93,102.1785","127,964.1959","33,409.5614",0.0633,0.0165,core
33,"Arts, Media & Services",5,34,Aliquam Auctor Associates,"92,443.4001","95,973.3005","2,991.6651",0.0420,0.0013,core
34,"Arts, Media & Services",6,35,Euismod In Corp.,"92,302.9490","95,744.7926","2,900.9453",0.0419,0.0013,core
39,"Arts, Media & Services",7,40,Pharetra Corp.,"82,506.9388","95,039.3014","11,631.3051",0.0631,0.0077,core
41,"Arts, Media & Services",8,42,Rutrum Magna Cras LLC,"79,547.3322","85,233.4311","5,068.1241",0.0373,0.0022,core
43,"Arts, Media & Services",9,44,At Pede Inc.,"72,957.8166","75,594.4060","2,081.5114",0.0615,0.0017,core
57,"Arts, Media & Services",10,58,Dolor Dolor Industries,"61,236.8301","63,344.1570","1,627.2287",0.0658,0.0017,core


## Interesting Merchants

In [44]:
top_ids = set(out.merchant_abn.head(TOP_K))
no_fraud_top = set(build_scores(lgd=0.0)["score"].nlargest(TOP_K).index)
gmv_top = set(FORECASTS[BEST_MODEL].nlargest(TOP_K).index)
o = out.set_index("merchant_abn")
fraud_trap = o[o.index.isin(no_fraud_top - top_ids)].nlargest(10, "exp_fraud_cost")
thin = o[o.index.isin(gmv_top - top_ids) & (o.take_rate < out.take_rate.median())
         & (o.exp_fraud_cost < 0.5 * o.exp_revenue)].nlargest(10, "forecast_gmv_12w")
picks = [
    ("rising star", o[(o.customers_12w >= 100) & ~o.index.isin(top_ids)].nlargest(10, "growth_12w"),
     lambda r: f"12w GMV {np.expm1(r.growth_12w):+.0%} vs prior 12w; rank {r['rank']}"),
    ("fraud trap", fraud_trap,
     lambda r: f"fraud cost ${r.exp_fraud_cost:,.0f} = {r.exp_fraud_cost / r.exp_revenue:.0%} of revenue; rank {r['rank']}"),
    ("big sales, thin margin", thin,
     lambda r: f"forecast GMV ${r.forecast_gmv_12w:,.0f} but take rate {r.take_rate:.2%}; rank {r['rank']}"),
    ("on the edge", o[o.p_top100_bootstrap.between(0.2, 0.8)].sort_values("rank").head(10),
     lambda r: f"P(top 100) {r.p_top100_bootstrap:.0%}, tier {r.robustness}; rank {r['rank']}"),
]
interesting = pd.concat([d.assign(reason=d.apply(fn, axis=1), group=g) for g, d, fn in picks])
interesting = interesting.reset_index()[["group", "merchant_abn", "name", "segment", "rank", "reason", "score",
                                         "exp_revenue", "exp_fraud_cost", "forecast_gmv_12w", "take_rate", "growth_12w", "customers_12w"]]
interesting.to_csv(OUT / "interesting_merchants.csv", index=False)
with pd.option_context("display.max_colwidth", 80):
    display(interesting[["group", "name", "segment", "reason"]])

,group,name,segment,reason
0,rising star,Risus Donec Egestas Corp.,Electronics & Digital,12w GMV +82% vs prior 12w; rank 2521
1,rising star,Laoreet Institute,Fashion & Lifestyle,12w GMV +64% vs prior 12w; rank 1432
2,rising star,Et Foundation,Fashion & Lifestyle,12w GMV +61% vs prior 12w; rank 1399
3,rising star,Egestas Nunc Industries,Electronics & Digital,12w GMV +59% vs prior 12w; rank 2263
4,rising star,Vel Incorporated,"Arts, Media & Services",12w GMV +58% vs prior 12w; rank 1050
5,rising star,Vestibulum Ante Corporation,"Arts, Media & Services",12w GMV +57% vs prior 12w; rank 2835
6,rising star,Et Eros LLC,Fashion & Lifestyle,12w GMV +57% vs prior 12w; rank 1257
7,rising star,Nec Limited,Fashion & Lifestyle,12w GMV +56% vs prior 12w; rank 1144
8,rising star,Quis Pede Corporation,Electronics & Digital,12w GMV +56% vs prior 12w; rank 1408
9,rising star,Aliquet Metus Ltd,Fashion & Lifestyle,12w GMV +55% vs prior 12w; rank 1988


In [45]:
print("Files written to", OUT)
for p in sorted(OUT.iterdir()):
    print(f"  {p.name}")

Files written to /Users/leopham/Documents/dataset1_preprocessing/MAST30034_Project_2/outputs/ranking
  all_merchants_scored.csv
  backtest_results.csv
  fraud_estimate_validation.csv
  head_to_head_backtest.csv
  interesting_merchants.csv
  loyalty_check.csv
  robustness_scenarios.csv
  segment_summary.csv
  top100_detailed.csv
  top100_merchants.csv
  top10_per_segment.csv
  top10_per_segment_detailed.csv


### Additional Socioeconomic Analysis

In [ ]:
# customer profile of the top 100 vs other merchants, from the curated data
CURATED = ROOT / "data" / "curated" / "curated_transactions"
SOCIO = ["state", "gender", "SA2_CODE_2021", "disadvantage_decile", "economic_resources_decile", "education_occupation_decile"]
DECILES = ["disadvantage_decile", "economic_resources_decile", "education_occupation_decile"]

top_ids = set(ranked.index[:TOP_K])
socio = pd.concat([pd.read_parquet(CURATED / f"{p}.parquet", columns=["merchant_abn", "dollar_value"] + SOCIO) for p in TX_PARTS])
socio = socio[socio["merchant_abn"].isin(merchants["merchant_abn"])]
socio["group"] = np.where(socio["merchant_abn"].isin(top_ids), "top 100", "other merchants")
socio["sa2_matched"] = socio["SA2_CODE_2021"].notna()       # unmatched postcodes have no area scores


# sales-weighted mean, ignoring rows with no area match
def weighted(d, cols):
    d = d[d["sa2_matched"]]
    return d[cols].mul(d["dollar_value"], axis=0).sum() / d["dollar_value"].sum()

profile = socio.groupby("group").apply(lambda d: weighted(d, DECILES))
profile["share_of_sales_with_area_match"] = socio.groupby("group").apply(lambda d: d.loc[d["sa2_matched"], "dollar_value"].sum() / d["dollar_value"].sum())
state_mix = pd.crosstab(socio["group"], socio["state"], values=socio["dollar_value"], aggfunc="sum", normalize="index")
gender_mix = pd.crosstab(socio["group"], socio["gender"], values=socio["dollar_value"], aggfunc="sum", normalize="index")

customer_profile = pd.concat({"area deciles": profile, "state share": state_mix, "gender share": gender_mix}, axis=1).round(3)
customer_profile.to_csv(OUT / "top100_customer_profile.csv")

# per-merchant customer profile
m_sales = socio.groupby("merchant_abn")["dollar_value"].sum()
matched = socio[socio["sa2_matched"]]
m_profile = (matched[DECILES].mul(matched["dollar_value"], axis=0)
             .groupby(matched["merchant_abn"]).sum()
             .div(matched.groupby("merchant_abn")["dollar_value"].sum(), axis=0))
m_profile["share_of_sales_with_area_match"] = matched.groupby("merchant_abn")["dollar_value"].sum() / m_sales
m_profile["top_state"] = socio.groupby(["merchant_abn", "state"])["dollar_value"].sum().unstack().idxmax(axis=1)
m_profile = (ranked[["rank", "segment", "score", "take_rate", "fraud_share"]].join(m_profile)
             .rename_axis("merchant_abn").reset_index().round(4))
m_profile.to_csv(OUT / "merchant_customer_profile.csv", index=False)

del socio
customer_profile.T

/var/folders/jl/tmc9_s297kz10_rqzzc6xjfc0000gn/T/ipykernel_82321/3464154413.py:18: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  profile = socio.groupby("group").apply(lambda d: weighted(d, DECILES))
/var/folders/jl/tmc9_s297kz10_rqzzc6xjfc0000gn/T/ipykernel_82321/3464154413.py:19: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  profile["share_of_sales_with_area_match"] = socio.groupby("group").apply(lambda d

group                                        other merchants  top 100
area deciles disadvantage_decile                      4.9290   4.9320
             economic_resources_decile                5.2330   5.2340
             education_occupation_decile              4.7680   4.7720
             share_of_sales_with_area_match           0.8350   0.8360
state share  ACT                                      0.0090   0.0090
             NSW                                      0.2890   0.2880
             NT                                       0.0140   0.0140
             QLD                                      0.1480   0.1480
             SA                                       0.1140   0.1140
             TAS                                      0.0370   0.0370
             VIC                                      0.2310   0.2310
             WA                                       0.1580   0.1580
gender share Female                                   0.4440   0.4450
             Male                                     0.4540   0.4540
             Undisclosed                              0.1020   0.1020